In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
def tsm_lin(q):
    return q / q.sum()

def tsm_sm(q, scale):
    return F.softmax(scale * 2 * q, dim=0)

def get_p_opt(y, alpha, n_iter=40):
    y = y.detach().double()
    alpha = torch.as_tensor(alpha, dtype=torch.float64, device=y.device).detach()

    log_y = torch.where(y > 0, torch.log(y), -torch.inf)

    # Solve p* = clamp(y, λ, exp(2α)λ), with sum(p*) = 1
    # using η = log(λ)
    log_n = torch.log(torch.tensor(y.numel(), dtype=torch.float64, device=y.device))

    lo = -log_n - 2 * alpha   # sum(p*) <= 1
    hi = -log_n               # sum(p*) >= 1

    for _ in range(n_iter):
        eta = (lo + hi) / 2

        log_p = torch.clamp(log_y, min=eta, max=eta + 2 * alpha)
        mass = torch.exp(log_p).sum()

        if mass < 1:
            lo = eta
        else:
            hi = eta

    eta = (lo + hi) / 2
    log_p_opt = torch.clamp(log_y, min=eta, max=eta + 2 * alpha)

    return torch.exp(log_p_opt)

## 2-component

In [ ]:
SCALE = 30

q = torch.tensor([1, 1, 0, 0], dtype=torch.float)
y = tsm_lin(q)

s = torch.tensor([0.9, 0.9, -0.9, -0.9], dtype=torch.float)
z = SCALE * s
p = F.softmax(z, dim=0)

p_opt = get_p_opt(y, SCALE)

dL_dalpha        = (p - y) * s
dL_dalpha_struct = (p - p_opt) * s
dL_dalpha_res    = (p_opt - y) * s

# "soft positive-mass vs. negative-mass attribution" (generalizes over binary)

dL_dalpha_pos        = q * dL_dalpha
dL_dalpha_struct_pos = q * dL_dalpha_struct
dL_dalpha_res_pos    = q * dL_dalpha_res

dL_dalpha_neg        = (1 - q) * dL_dalpha
dL_dalpha_struct_neg = (1 - q) * dL_dalpha_struct
dL_dalpha_res_neg    = (1 - q) * dL_dalpha_res


print("y:                   ", y)
print("p:                   ", p)
print("p_opt:               ", p_opt)
print("dL_dz:               ", p - y)
print("dL_dalpha:           ", dL_dalpha)

print("dL_dalpha_struct:    ", dL_dalpha_struct)
print("dL_dalpha_res:       ", dL_dalpha_res)

print("dL_dalpha_struct_pos:", dL_dalpha_struct_pos)
print("dL_dalpha_res_pos:   ", dL_dalpha_res_pos)

print("dL_dalpha_struct_neg:", dL_dalpha_struct_neg)
print("dL_dalpha_res_neg:   ", dL_dalpha_res_neg)

## 3-component

In [ ]:
# SCALE = 100
SCALE = 30
# SCALE = 10
# SCALE = 3
# SCALE = 1

q = torch.tensor([1, 1, 0, 0], dtype=torch.float)
# q = torch.tensor([1, 1, 0.1, 0.1], dtype=torch.float)
# q = torch.tensor([1, 1, 0.5, 0.0], dtype=torch.float)

# y = tsm_lin(q)
y = tsm_sm(q, SCALE)

# s = torch.tensor([0.8, 0.8, 0.2, 0.0], dtype=torch.float)
s = torch.tensor([1, 0.8, -1, -1], dtype=torch.float)
z = SCALE * s
p = F.softmax(z, dim=0)

dL_dalpha = (p - y) * s


p_opt = get_p_opt(y, SCALE)

# For full-support soft targets, use inverse-softmax target logits to compute s_opt:
log_p_opt = torch.log(p_opt)
# z_opt = log_p_opt - 0.5 * (log_p_opt.max() + log_p_opt.min())
z_opt = log_p_opt - 0.5 * (log_p_opt.max(dim=-1, keepdim=True).values + log_p_opt.min(dim=-1, keepdim=True).values)  # for row-wise softmax: compute the min/max per row, not globally across the whole batch
s_opt = z_opt / SCALE


dL_dalpha_struct = (p - p_opt) * s
dL_dalpha_sres = (p_opt - y) * (s - s_opt)
dL_dalpha_ires = (p_opt - y) * s_opt
dL_dalpha_res = dL_dalpha_sres + dL_dalpha_ires

print("2alpha:     ", 2 * SCALE)
print("logyminymax:", torch.log(y.max() / y.min()).item())
print("---")
print("q:               ", q)
print("y:               ", y)
print("s:               ", s)
print("p:               ", p)
print("p_opt:           ", p_opt)
print("s_opt:           ", s_opt)
print("dL_dz:           ", p - y)
print("dL_dalpha:       ", dL_dalpha)
print("---")
print("dL_dalpha_struct:", dL_dalpha_struct)
print("dL_dalpha_res:   ", dL_dalpha_res)
print("dL_dalpha_sres:  ", dL_dalpha_sres)
print("dL_dalpha_ires:  ", dL_dalpha_ires)